In [1]:
!pip install -q spacy sentence-transformers transformers torch scikit-learn geopy
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
      --------------------------------------- 0.5/33.5 MB 3.8 MB/s eta 0:00:09
      --------------------------------------- 0.5/33.5 MB 3.8 MB/s eta 0:00:09
      --------------------------------------- 0.8/33.5 MB 1.1 MB/s eta 0:00:30
      --------------------------------------- 0.8/33.5 MB 1.1 MB/s eta 0:00:30
      --------------------------------------- 0.8/33.5 MB 1.1 MB/s eta 0:00:30
     - ------------------------------------- 1.0/33.5 MB 747.5 kB/s eta 0:00:44
     - ------------------------------------- 1.0/33.5 MB 747.5 kB/s eta 0:00:44
     - ------------------------------------- 1.3/33.5 MB 744.6 kB/s eta 0:00:44
     -- ------------------------------------ 1.8/33.5 MB 888.8 kB/s eta 0:00:36
     -- ------------------------------------ 2.1/33.5 MB 927.0 kB/s eta 0:00:34
     -- ------------------------------------ 2.1/33.5 MB 927.

In [2]:
import pandas as pd
import numpy as np
import spacy
import torch
import re
import time
import requests

from datetime import datetime
from functools import lru_cache
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import pipeline

In [3]:
nlp = spacy.load("en_core_web_md")
semantic_model = SentenceTransformer('all-mpnet-base-v2')
geolocator = Nominatim(user_agent="forensic_ai")

# Pretrained Fake News BERT (NO TRAINING NEEDED)
bert_model = pipeline(
    "text-classification",
    model="mrm8488/bert-tiny-finetuned-fake-news-detection"
)

print("All models loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mrm8488/bert-tiny-finetuned-fake-news-detection
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All models loaded


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

In [5]:
@lru_cache(maxsize=500)
def get_coords(location):
    try:
        time.sleep(1)
        loc = geolocator.geocode(location)
        return (loc.latitude, loc.longitude) if loc else (None, None)
    except:
        return (None, None)


def extract_atomic_claims(text):
    doc = nlp(text)
    claims = []
    locations = []
    
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC"]:
            locations.append(ent.text)
    
    for sent in doc.sents:
        subj, verb, obj = "", "", ""
        
        for token in sent:
            if token.dep_ == "nsubj":
                subj = token.text
            if token.pos_ == "VERB":
                verb = token.lemma_
            if token.dep_ in ["dobj", "pobj"]:
                obj = token.text
        
        if subj and verb and obj:
            claims.append(f"{subj} {verb} {obj}")
    
    return claims, locations


def semantic_score(text, kb):
    return util.cos_sim(
        semantic_model.encode(text),
        semantic_model.encode(kb)
    ).max().item()


def spatial_check(loc, coords):
    if not loc or coords == (None, None):
        return False
    expected = get_coords(loc)
    if expected == (None, None):
        return False
    return geodesic(expected, coords).km > 150


def temporal_check(d1, d2):
    try:
        d1 = datetime.strptime(d1, "%Y-%m-%d")
        d2 = datetime.strptime(d2, "%Y-%m-%d")
        return abs((d1 - d2).days) > 30
    except:
        return False
def reverse_image_search_simulator(text):
    
    strong_fake_patterns = [
        "shark swimming", 
        "viral hoax", 
        "fake image", 
        "edited photo"
    ]
    
    for phrase in strong_fake_patterns:
        if phrase in text.lower():
            return True, "Recycled viral image detected"
    
    return False, "No recycled media detected"
def get_news_source(query):
    try:
        API_KEY = "a696850430fe42aaaeeeefeeece8729f"   # ← paste your real key here
        url = f"https://newsapi.org/v2/everything?q={query}&apiKey={API_KEY}"
        response = requests.get(url).json()
        
        if response["status"] == "ok" and response["totalResults"] > 0:
            article = response["articles"][0]
            return {
                "source": article["source"]["name"],
                "url": article["url"]
            }
        else:
            return None
    except:
        return None

In [6]:
def forensic_system(text, meta, kb):
    
    claims, locs = extract_atomic_claims(text)
    claim = claims[0] if claims else text
    
    sim = semantic_score(claim, kb)
    
    verdict = 1
    reasons = []
    
    # Semantic Check
    if sim < 0.15:
        verdict = 0
        reasons.append(f"Low semantic similarity ({sim:.2f})")
    
    # Spatial Check
    if locs and spatial_check(locs[0], meta['coords']):
        verdict = 0
        reasons.append("Location mismatch detected")
    
    # Temporal Check
    if temporal_check(meta['news_date'], meta['img_date']):
        verdict = 0
        reasons.append("Recycled media (time mismatch)")
    
    # Reverse Image Check (NEW)
    recycled, msg = reverse_image_search_simulator(text)
    if recycled:
        verdict = 0
        reasons.append(msg)
    
    return verdict, reasons

In [7]:
df = pd.read_csv("posts.txt", sep="\t")

# BALANCE DATASET
df_real = df[df['label'] == 'real']
df_fake = df[df['label'] == 'fake']

min_size = min(len(df_real), len(df_fake))

df_balanced = pd.concat([
    df_real.sample(min_size, random_state=42),
    df_fake.sample(min_size, random_state=42)
])

df = df_balanced.sample(frac=1).reset_index(drop=True)

df['clean'] = df['post_text'].apply(clean_text)

vectorizer = TfidfVectorizer(max_features=8000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['clean'])

y = df['label'].apply(lambda x: 1 if x == 'real' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

ml_model = LogisticRegression(max_iter=1000, class_weight='balanced')
ml_model.fit(X_train, y_train)

preds = ml_model.predict(X_test)

print("ML Accuracy:", accuracy_score(y_test, preds)*100)

ML Accuracy: 88.83534136546186


In [8]:
def bert_predict(text):
    result = bert_model(text[:512])[0]
    return 1 if result['label'] == 'REAL' else 0, result['score']

In [9]:
def final_system(text, meta, kb):
    
    # --- ML ---
    vec = vectorizer.transform([clean_text(text)])
    ml_pred = ml_model.predict(vec)[0]
    
    # --- BERT ---
    bert_pred, bert_conf = bert_predict(text)
    
    # --- Forensic ---
    f_pred, f_reasons = forensic_system(text, meta, kb)
    
    reasons = []
    source_info = get_news_source(text)
    
    # STEP 1: KNOWLEDGE MATCH (VERY IMPORTANT)
    for fact in kb:
        if fact.lower() in text.lower():
            reasons.append("Matched trusted knowledge base")
            return 1, reasons, source_info
    
    # STEP 2: BERT confident REAL
    if bert_pred == 1 and bert_conf > 0.6:
        reasons.append(f"BERT confident REAL ({bert_conf:.2f})")
        return 1, reasons, source_info
    
    # STEP 3: BOTH say FAKE
    if ml_pred == 0 and bert_pred == 0:
        reasons.append("ML + BERT indicate FAKE")
        return 0, reasons, source_info
    
    # STEP 4: FORENSIC CHECK
    if f_pred == 0:
        reasons.extend(f_reasons)
        return 0, reasons, source_info
    
    # STEP 5: DEFAULT REAL
    reasons.append("No strong fake signals → REAL")
    return 1, reasons, source_info

In [10]:
kb = [
    "The Eiffel Tower is located in Paris, France.",
    "The Oscars are held annually to honor films.",
    "New York is a major city in the United States.",
    "Earthquakes can cause severe damage.",
    "Protests often occur in major cities.",
    "Fake viral images are often edited digitally.",
    "Shark photos in flooded streets are fake.",
    "Nepal earthquake in 2015 caused destruction.",
    "London is the capital of the United Kingdom.",
    "Paris is the capital of France."
]

In [11]:
test_cases= [

    # ✅ REAL CASES
    {
        "text": "Paris is the capital of France.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (48.8566, 2.3522)}
    },
    {
        "text": "London is the capital of the United Kingdom.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (51.5074, -0.1278)}
    },
    {
        "text": "The 2015 Nepal earthquake caused major destruction.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (27.7172, 85.3240)}
    },

    # ❌ FAKE CASES
    {
        "text": "Humans can breathe in space without oxygen.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (0.0, 0.0)}
    },
    {
        "text": "The Sun rises in the west every day.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (0.0, 0.0)}
    },
    {
        "text": "Dinosaurs are still alive and living in Africa.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (1.2921, 36.8219)}
    },

    # ⚠️ TEMPORAL (RECYLED MEDIA)
    {
        "text": "A massive flood is currently happening in Sydney.",
        "meta": {"news_date": "2026-04-01", "img_date": "2015-05-01", "coords": (-33.8688, 151.2093)}
    },
    {
        "text": "A viral image shows a giant wave hitting Sydney.",
        "meta": {"news_date": "2026-04-01", "img_date": "2015-06-01", "coords": (-33.8688, 151.2093)}
    },

    # ⚠️ SPATIAL MISMATCH
    {
        "text": "The Eiffel Tower is located in Paris.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (40.7128, -74.0060)}  # New York coords ❌
    },
    {
        "text": "The Taj Mahal is located in India.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-04-01", "coords": (51.5074, -0.1278)}  # London coords ❌
    },

    # 🧠 MIXED / TRICKY
    {
        "text": "A video shows a man jumping into Mount Fuji volcano.",
        "meta": {"news_date": "2026-04-01", "img_date": "2010-01-01", "coords": (35.3606, 138.7274)}
    },
    {
        "text": "Protests are happening in London today.",
        "meta": {"news_date": "2026-04-01", "img_date": "2026-03-30", "coords": (51.5074, -0.1278)}
    }

]
for t in test_cases:
    pred, reasons, source = final_system(t["text"], t["meta"], kb)
    
    print("\n NEWS ANALYSIS REPORT")
    print("Text:", t["text"])
    print("Verdict:", "REAL" if pred==1 else "FAKE")
    
    print("Rationale:")
    for r in reasons:
        print(" -", r)
    if source:
        print("Source:", source["source"])
        print("Link:", source["url"])
    else:
        print("No reliable source found")


 NEWS ANALYSIS REPORT
Text: Paris is the capital of France.
Verdict: REAL
Rationale:
 - Matched trusted knowledge base
Source: Themarginalian.org
Link: https://www.themarginalian.org/2026/04/04/mandelbrot-fractals-chaos/

 NEWS ANALYSIS REPORT
Text: London is the capital of the United Kingdom.
Verdict: REAL
Rationale:
 - Matched trusted knowledge base
Source: Openculture.com
Link: https://www.openculture.com/2026/04/explore-1000000-digitized-artworks-from-across-the-uk.html

 NEWS ANALYSIS REPORT
Text: The 2015 Nepal earthquake caused major destruction.
Verdict: REAL
Rationale:
 - No strong fake signals → REAL
No reliable source found

 NEWS ANALYSIS REPORT
Text: Humans can breathe in space without oxygen.
Verdict: REAL
Rationale:
 - No strong fake signals → REAL
Source: Raw Story
Link: https://www.rawstory.com/return-to-earth-at-this-kansas-space-museum-i-came-in-search-of-what-we-had-nearly-lost/

 NEWS ANALYSIS REPORT
Text: The Sun rises in the west every day.
Verdict: REAL
Rationa